# 04 Orchestrated run (scheduled by the Fabric Data Pipeline)
Runs all stages in order with logging, retries, the quality gate and reconciliation. Raises an error when the run fails, so the pipeline activity turns red and its failure path (alert email) fires.

In [ ]:
# Requires: this notebook attached to the lakehouse 'lh_mfg_ops' (default lakehouse)
# and the Fabric environment 'env_mfgops', which has the mfgops wheel from CI installed.
from mfgops.lakehouse import Lakehouse
lh = Lakehouse.detect(spark)
print("Running in Fabric:", lh.fabric)

In [ ]:
from mfgops.orchestrate import run_pipeline
res = run_pipeline(lh, max_error_rate=0.02)
print(res.run_id, res.status, res.failed_stage, res.message)

In [ ]:
display(lh.read('meta', 'pipeline_runs').filter(f"run_id = '{res.run_id}'").orderBy('started_at'))

In [ ]:
if res.status != 'success':
    raise RuntimeError(f'Pipeline run {res.run_id} failed at {res.failed_stage}: {res.message}')